In [1]:
from qiskit.quantum_info import random_unitary
from qiskit.circuit.library import UnitaryGate
from qiskit import QuantumCircuit, transpile
from qiskit_aer.noise import NoiseModel, depolarizing_error, QuantumError
from qiskit.quantum_info import Operator
from math import comb
from qiskit_aer import AerSimulator
import math
import matplotlib.pyplot as plt
import random
import pickle
from qiskit import QuantumCircuit
from qiskit.circuit.library.standard_gates import IGate, XGate, YGate, ZGate
import numpy as np

In [2]:
# a rudimentary construction of Pauli twirling embedding
PAULI_GATES = {'I': IGate(), 'X': XGate(), 'Y': YGate(), 'Z': ZGate()}
CX_PAULI_TWIRLING_LOOKUP = {
    ('I', 'I'): ('I', 'I'),
    ('I', 'X'): ('I', 'X'),
    ('I', 'Y'): ('Z', 'Y'),
    ('I', 'Z'): ('Z', 'Z'),
    ('X', 'I'): ('X', 'X'),
    ('X', 'X'): ('X', 'I'),
    ('X', 'Y'): ('Y', 'Z'),
    ('X', 'Z'): ('Y', 'Y'),
    ('Y', 'I'): ('Y', 'X'),
    ('Y', 'X'): ('Y', 'I'),
    ('Y', 'Y'): ('X', 'Z'),
    ('Y', 'Z'): ('X', 'Y'),
    ('Z', 'I'): ('Z', 'I'),
    ('Z', 'X'): ('Z', 'X'),
    ('Z', 'Y'): ('I', 'Y'),
    ('Z', 'Z'): ('I', 'Z'),
}

def apply_cx_pauli_twirling(circuit: QuantumCircuit, random_seed: int) -> QuantumCircuit:
    """
    Returns a new circuit with Pauli twirling applied to each CX gate. ASSUMES ALL QUBITS
    ARE MEASURED IN ORDER
    """
    np.random.seed(random_seed)
    new_circ = QuantumCircuit(circuit.num_qubits)
    
    for instr, qargs, cargs in circuit.data:
        if instr.name == 'cx':
            # Randomly choose Pauli gates for both qubits
            pauli_1 = np.random.choice(['I', 'X', 'Y', 'Z'])
            pauli_2 = np.random.choice(['I', 'X', 'Y', 'Z'])
            
            # Apply pre-twirl Paulis
            new_circ.append(PAULI_GATES[pauli_1], [qargs[0]._index])
            new_circ.append(PAULI_GATES[pauli_2], [qargs[1]._index])

            pauli_1_prime, pauli_2_prime = CX_PAULI_TWIRLING_LOOKUP[(pauli_1, pauli_2)]
            
            # Apply CX gate
            new_circ.cx(qargs[0]._index, qargs[1]._index)
            
            # Apply post-twirl correction (same Pauli again)
            new_circ.append(PAULI_GATES[pauli_1_prime], [qargs[0]._index])
            new_circ.append(PAULI_GATES[pauli_2_prime], [qargs[1]._index])
        else:
            if instr.name.lower() != 'measure':
                new_circ.append(instr, [q._index for q in qargs], cargs)
    new_circ.measure_all()
    
    return new_circ

# compose multiple counts lists into a single dictionary
def compose_counts(counts_list: list[dict[str, int]]) -> dict[str, int]:
    output_counts = {}
    for counts in counts_list:
        for bitstring, count in counts.items():
            if bitstring not in output_counts:
                output_counts[bitstring] = count
            else:
                output_counts[bitstring] += count
    return output_counts

# marginalize a counts distribution against total shots
def counts_to_probs(counts):
    return {bitstring:counts[bitstring]/sum(list(counts.values())) for bitstring in counts}

# module for error mitigation code
import numpy as np

# invert depolarizing noise to construct a new pseudodistribution
def invert_depolarizing_noise(probs: dict[str, float], depolarizing_error: float, num_gates: int):
    # invert the depolarizing noise for a single bitstring probability
    def invert_depolarizing_noise_prob(prob: float, depolarizing_error: float, num_gates: int, num_qubits: int):
        return prob/((1 - depolarizing_error) ** num_gates) - (1 - (1 - depolarizing_error) ** num_gates) / ((2**num_qubits) * ((1-depolarizing_error)**num_gates))
    return {bitstring:invert_depolarizing_noise_prob(prob, depolarizing_error, num_gates, len(list(probs.keys())[0])) for (bitstring, prob) in probs.items()}

def projection_simplex_sort(pseudo_probs, z=1):
    """
    Projects a vector v onto the probability simplex (sum to z, non-negative).

    Args:
        v (np.array): The input vector to project.
        z (float): The target sum for the projected vector (default is 1 for probability simplex).

    Returns:
        np.array: The projected vector.
    """
    bitstrings = list(pseudo_probs.keys())
    v = np.array([pseudo_probs[bitstring] for bitstring in pseudo_probs])
    n_features = v.shape[0]

    # Sort the input vector in descending order
    u = np.sort(v)[::-1]

    # Compute the cumulative sum of sorted elements and subtract z
    cssv = np.cumsum(u) - z

    # Find the index rho where the condition u - cssv / (index + 1) > 0 is met
    ind = np.arange(n_features) + 1
    cond = u - cssv / ind > 0
    
    # Handle the case where no element satisfies the condition (e.g., all elements are negative)
    if not np.any(cond):
        rho = 0  # Default to the first element if no condition is met
        theta = 0.0
    else:
        rho = ind[cond][-1]
        theta = cssv[cond][-1] / float(rho)

    # Compute the projected vector
    w = np.maximum(v - theta, 0)
    return {bitstrings[i]:w[i] for i in range(len(bitstrings))}



In [3]:
random_seed = 1234

In [4]:
random.seed(random_seed)
np.random.seed(random_seed)

In [5]:
def generate_approx_haar_circuit(n, depth, seed, basis_gates=['cx', 'x', 'sx', 'rz', 'id'], optimization_level=3):
    """
    Generates an n-qubit random quantum circuit to approximate a unitary 2-design
    on an all-to-all connected hardware graph.
    """
    qc = QuantumCircuit(n)
    
    for d in range(depth):
        # --- LAYER 1: Single-qubit Haar-random gates ---
        for i in range(n):
            # Sample a 1-qubit Haar random unitary
            u_1q = random_unitary(2, seed=(seed + 100*d)).to_instruction()
            qc.append(u_1q, [i])
        
        # --- LAYER 2: Entangling layer (All-to-all) ---
        # Generate a random perfect matching for the n qubits
        qubits = list(range(n))
        random.shuffle(qubits)
        
        # Apply 2-qubit gates to the random pairs
        for i in range(0, n - 1, 2):
            q1, q2 = qubits[i], qubits[i+1]
            
            u_2q = random_unitary(4, seed=(seed + 100*d)).to_instruction()
            qc.append(u_2q, [q1, q2])
    #qc.measure_all()
    transpiled_circ = transpile(qc, basis_gates=basis_gates, optimization_level=optimization_level)
    transpiled_circ.save_state()
            
    return transpiled_circ

In [6]:
def count_cx(target_pair, circuit):
    cx_count = 0
    for instruction in circuit.data:
        if target_pair[0] in [qubit._index for qubit in instruction.qubits] and target_pair[1] in [qubit._index for qubit in instruction.qubits] and instruction.operation.name == 'cx':
            cx_count += 1
    return cx_count

def count_cx_all_pairs(circuit, qubit_count):
    cx_counts_by_pair = []
    for q0 in range(qubit_count):
        for q1 in range(q0+1, qubit_count):
            cx_counts_by_pair.append(count_cx([q0,q1], circuit))
    return np.array(cx_counts_by_pair)

In [7]:
num_qubits = 10
num_circuits = 10
depth = 150
haar_random_circuits = [generate_approx_haar_circuit(num_qubits, depth, seed=seed) for seed in range(num_circuits)]

In [8]:
# populate G matrix
def get_g_matrix(circuit):
    G = np.zeros(shape=(num_qubits, num_qubits))
    for q0 in range(num_qubits):
        for q1 in range(q0+1, num_qubits):
            G[q0,q1] = count_cx([q0,q1], circuit)
            G[q1,q0] = G[q0,q1]
    return G

In [9]:
twoq_depolarizing_errors = 0.001 * np.random.rand(comb(num_qubits, 2))
index = 0
E = np.zeros(shape=(num_qubits, num_qubits))
for q0 in range(num_qubits):
    for q1 in range(q0 + 1, num_qubits):
        E[q0,q1] = 1 - twoq_depolarizing_errors[index]
        E[q1,q0] = 1 - twoq_depolarizing_errors[index]
        # Depolarizing quantum error
        cx_depolarizing_error = depolarizing_error(twoq_depolarizing_errors[index], 2)
        index += 1
ideal_simulator = AerSimulator(noise_model=None, method="density_matrix")

In [10]:
def noisy_simulator(twoq_depolarizing_errors, num_qubits, xx_error_strength=0):
    index = 0
    noise_model = NoiseModel()
    coherent_unitary = np.array(
        [
            [np.cos(xx_error_strength/2), 0, 0, -1j*np.sin(xx_error_strength/2)],
            [0, np.cos(xx_error_strength/2), -1j*np.sin(xx_error_strength/2), 0],
            [0, -1j*np.sin(xx_error_strength/2), np.cos(xx_error_strength/2), 0],
            [-1j*np.sin(xx_error_strength/2), 0, 0, np.cos(xx_error_strength/2)]
        ]
    )
    coherent_error_op = Operator(coherent_unitary)

    # Create a QuantumError from the coherent unitary
    # The error is applied with probability 1.0 for a coherent error
    coherent_error = QuantumError([(coherent_error_op, 1.0)])

    for q0 in range(num_qubits):
        for q1 in range(q0+1, num_qubits):
            cx_depolarizing_error = depolarizing_error(twoq_depolarizing_errors[index],2)
            noise_model.add_quantum_error(cx_depolarizing_error, ['cx'], [q0, q1])
            noise_model.add_quantum_error(cx_depolarizing_error, ['cx'], [q1, q0])
            noise_model.add_quantum_error(coherent_error, ['cx'], [q0,q1])
            noise_model.add_quantum_error(coherent_error, ['cx'], [q1,q0])
            index += 1
    return AerSimulator(noise_model=noise_model, method='density_matrix')



In [11]:
def offdiag_variance(F, E, G):
    """Variance of E[F(i),F(j)]**G[i,j] over i != j."""
    X = E[F][:, F] ** G
    n = len(F)
    
    mask = ~np.eye(n, dtype=bool)
    vals = X[mask]
    
    return np.var(vals)


def simulated_annealing(E, G, steps=5000, T0=1.0, seed=None):
    """
    Minimize off-diagonal variance using simple simulated annealing.
    Suitable for n < 10.
    """
    if seed is not None:
        np.random.seed(seed)

    n = len(E)
    F = np.random.permutation(n)
    best_F = F.copy()
    best_val = offdiag_variance(F, E, G)

    for t in range(steps):
        T = T0 * (1 - t / steps)  # linear cooling
        
        # propose swap
        i, j = np.random.choice(n, 2, replace=False)
        F_new = F.copy()
        F_new[i], F_new[j] = F_new[j], F_new[i]
        
        new_val = offdiag_variance(F_new, E, G)
        delta = new_val - best_val
        
        # accept move?
        if delta < 0 or np.random.rand() < np.exp(-delta / max(T, 1e-12)):
            F = F_new
            if new_val < best_val:
                best_val = new_val
                best_F = F.copy()

    best_mapping = {logical_qubit:int(best_F[logical_qubit]) for logical_qubit in range(num_qubits)}
    return best_mapping, best_val

In [12]:
def reindex_circuit(circ_orig, new_qubit_map):
    circ_new = QuantumCircuit(num_qubits)
    for instruction in circ_orig.data:
        op = instruction.operation
        # Get old qubits and map them to new indices
        old_qubits = instruction.qubits
        new_qubits_indices = [new_qubit_map[q._index] for q in old_qubits]
        if op.name != 'measure':
            circ_new.append(op, new_qubits_indices, instruction.clbits)
    return circ_new

In [13]:
haar_random_circuits_layout_selected = []
annealing_objectives = []
for circuit in haar_random_circuits:
    best_mapping, best_cost = simulated_annealing(E, get_g_matrix(circuit))
    haar_random_circuits_layout_selected.append(reindex_circuit(circuit, best_mapping))
    annealing_objectives.append(best_cost)

In [14]:
def count_equivalent_cx_gates(circuit, pair, E, target_lambda):
    num_gates = count_cx(pair, circuit)
    pair_lambda = E[pair[0], pair[1]]
    equivalent_gate_count = num_gates * np.log(pair_lambda) / np.log(target_lambda)
    return equivalent_gate_count
def count_total_equivalent_cx_gates(circuit, E, target_lambda):
    equivalent_gate_count = 0
    for q0 in range(num_qubits):
        for q1 in range(q0 + 1, num_qubits):
            equivalent_gate_count += count_equivalent_cx_gates(circuit, (q0, q1), E, target_lambda)
    return equivalent_gate_count

In [15]:
chosen_lambda = 0.999
twirl_counts = [1, 1, 4, 9]
xx_errors = [0, 3e-3, 6e-3, 9e-3]
estimated_global_depolarizing_parameter = (chosen_lambda ** (15/16))
index = 0

probs_unmitigated_per_coherent_error = []
probs_mitigated_per_coherent_error = []
for c, coherent_error in enumerate(xx_errors):
    twirls = twirl_counts[c]
    simulator = noisy_simulator(twoq_depolarizing_errors, num_qubits, coherent_error)
    result = simulator.run(haar_random_circuits_layout_selected).result()
    probs_per_circuit = [np.real(np.diag(result.data(i)['density_matrix'])) for i in range(num_circuits)]
    probs_per_circuit = [{f"{i:0{num_qubits}b}":prob for i, prob in enumerate(probs)} for probs in probs_per_circuit]
    probs_unmitigated_per_coherent_error.append(probs_per_circuit)

    probs_mitigated_per_circuit = []
    for circuit in haar_random_circuits_layout_selected:
        print(index)
        twirled_circuits = [apply_cx_pauli_twirling(circuit, twirling_seed) for twirling_seed in range(twirls)]
        result = simulator.run(twirled_circuits).result()
        probs_per_circuit = [np.real(np.diag(result.data(i)['density_matrix'])) for i in range(twirls)]
        probs_per_circuit = [{f"{i:0{num_qubits}b}":prob for i, prob in enumerate(probs)} for probs in probs_per_circuit]
        probs = counts_to_probs(compose_counts(probs_per_circuit))
        pseudo_probs = invert_depolarizing_noise(probs, 1 - estimated_global_depolarizing_parameter, count_total_equivalent_cx_gates(circuit, E, chosen_lambda))
        probs_mitigated_per_circuit.append(projection_simplex_sort(pseudo_probs))
        index += 1
    probs_mitigated_per_coherent_error.append(probs_mitigated_per_circuit)

0


C:\Users\ayush\AppData\Local\Temp\ipykernel_18544\1110102311.py:30: DeprecationWarning: Treating CircuitInstruction as an iterable is deprecated legacy behavior since Qiskit 1.2, and will be removed in Qiskit 3.0. Instead, use the `operation`, `qubits` and `clbits` named attributes.
  for instr, qargs, cargs in circuit.data:


1
2
3
4
5
6
7
8
9


10
11
12
13
14
15
16
17
18
19


20
21
22
23
24
25
26
27
28
29


30
31
32
33
34
35
36
37
38
39


In [16]:
result = ideal_simulator.run(haar_random_circuits_layout_selected).result()
ideal_probs_per_circuit = [np.real(np.diag(result.data(i)['density_matrix'])) for i in range(num_circuits)]
ideal_probs_per_circuit = [{f"{i:0{num_qubits}b}":prob for i, prob in enumerate(probs)} for probs in ideal_probs_per_circuit]

In [17]:
def hellinger_distance(P, Q):
    keys = set(P.keys()).union(Q.keys())
    return math.sqrt(
        0.5 * sum(
            (math.sqrt(P.get(k, 0.0)) - math.sqrt(Q.get(k, 0.0)))**2
            for k in keys
        )
    )

In [18]:
mitigated_median_infidelities = []
unmitigated_median_infidelities = []
median_infidelity_reductions = []
mitigated_infidelities = []
unmitigated_infidelities = []
for index in range(len(xx_errors)):
    coherent_error = xx_errors[index]
    mitigated_infidelity = np.array([hellinger_distance(ideal_probs_per_circuit[i], probs_mitigated_per_coherent_error[index][i]) for i in range(num_circuits)])
    mitigated_infidelities.append(mitigated_infidelity)
    median_mitigated_infidelity = np.median(mitigated_infidelity)
    unmitigated_infidelity = np.array([hellinger_distance(ideal_probs_per_circuit[i], probs_unmitigated_per_coherent_error[index][i]) for i in range(num_circuits)])
    unmitigated_infidelities.append(unmitigated_infidelity)
    median_unmitigated_infidelity = np.median(unmitigated_infidelity)
    infidelity_reduction = 100*(unmitigated_infidelity-mitigated_infidelity)/unmitigated_infidelity
    mitigated_median_infidelities.append(median_mitigated_infidelity)
    unmitigated_median_infidelities.append(median_unmitigated_infidelity)
    median_infidelity_reductions.append(np.median(infidelity_reduction))

In [19]:
mean_mitigated_pauli_twirls = []
mitigated_errors = []
mean_unmitigated = []
unmitigated_errors = []
unmitigated_errors_check = []
for i in range(len(xx_errors)):
    print(i)
    mean_mitigated_pauli_twirls.append(np.mean(mitigated_infidelities[i]))
    mitigated_errors.append(np.std(mitigated_infidelities[i])/np.sqrt(num_circuits))
    mean_unmitigated.append(np.mean(unmitigated_infidelities[i]))
    unmitigated_errors.append(np.std(unmitigated_infidelities[i])/np.sqrt(num_circuits))




0
1
2
3


In [20]:
print(mean_mitigated_pauli_twirls)

[np.float64(0.014644712951361277), np.float64(0.03642218793999841), np.float64(0.03546687713077196), np.float64(0.038685930922927644)]


In [21]:
print(mean_unmitigated)

[np.float64(0.2183638290291145), np.float64(0.21883732568042694), np.float64(0.22027002732460127), np.float64(0.22262136018289164)]
